# Historical Electoral College Data
https://en.wikipedia.org/wiki/United_States_Electoral_College#Chronological_table

In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import re

This notebook was created with the help of claude code, as a way to learn how to retreive data from websites. The following cells, created by claude, contain code and print statements using the `requests` library (print statements were used as a way to track progress and debug).

In [2]:
# hardcode url
url = 'https://en.wikipedia.org/wiki/United_States_Electoral_College'

# user-agent for Wikipedia's servers
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

print("Fetching Wikipedia page...")

# make request
response = requests.get(url, headers=headers)

# parse html
soup = BeautifulSoup(response.content, 'html.parser')
print("✓ Page loaded\n")

Fetching Wikipedia page...
✓ Page loaded



After the webpage has been loaded, now it's time to find the correct table to scrape, specifically the table with the historical electoral college votes for each state.

In [3]:
# print statement for progress and debugging
print("Looking for table with caption 'Number of presidential electors by state and year'...")

# create table variable to store future table data
table = None
for t in soup.find_all('table'):
    caption = t.find('caption')
    # hardcode caption of table (many different tables in webpage)
    if caption and 'presidential electors by state' in caption.get_text().lower():
        print(f"✓ Found table with caption: '{caption.get_text(strip=True)}'\n")
        table = t
        break

# print statement if table is not found
if not table:
    print("✗ Table not found by caption. Showing all table captions:")
    for i, t in enumerate(soup.find_all('table'), 1):
        caption = t.find('caption')
        if caption:
            print(f"  {i}. {caption.get_text(strip=True)}")
    exit()

Looking for table with caption 'Number of presidential electors by state and year'...
✓ Found table with caption: 'Number of presidential electors by state and year'



Return all rows from table of electoral college votes:

In [4]:
# 'tr' is table row in html; parse the rows
rows = table.find_all('tr')

# print statement to show the rows have been found
print(f"✓ Found {len(rows)} rows")

✓ Found 56 rows


Retrieve and save data from the rows:

In [5]:
# create list variable for data
all_data = []

# 'th' table header and 'td' table data
for i, row in enumerate(rows):
    cells = row.find_all(['th', 'td'])

    if cells:
        # Clean the text from each cell
        row_data = []
        for cell in cells:
            text = cell.get_text(strip=True)
            # Remove reference brackets like [1], [2]
            text = re.sub(r'\[.*?\]', '', text)
            row_data.append(text)

        all_data.append(row_data)

        # Print first few cells as preview (first few 'columns' all rows)
        preview = ' | '.join(row_data[:5])
        if len(row_data) > 5:
            preview += ' | ...'
        print(f"Row {i}: {preview}")

Row 0: Electionyear | 1788–1800 | 1804–1900 | 1904–2000 | 2004–
Row 1: '88 | '92 | '96'00 | '04'08 | '12 | ...
Row 2: # | Total | 81 | 135 | 138 | ...
Row 3:  | State |  |  |  | ...
Row 4: 22 | Alabama |  |  |  | ...
Row 5: 49 | Alaska |  |  |  | ...
Row 6: 48 | Arizona |  |  |  | ...
Row 7: 25 | Arkansas |  |  |  | ...
Row 8: 31 | California |  |  |  | ...
Row 9: 38 | Colorado |  |  |  | ...
Row 10: 5 | Connecticut | 7 | 9 | 9 | ...
Row 11: – | D.C. |  |  |  | ...
Row 12: 1 | Delaware | 3 | 3 | 3 | ...
Row 13: 27 | Florida |  |  |  | ...
Row 14: 4 | Georgia | 5 | 4 | 4 | ...
Row 15: 50 | Hawaii |  |  |  | ...
Row 16: 43 | Idaho |  |  |  | ...
Row 17: 21 | Illinois |  |  |  | ...
Row 18: 19 | Indiana |  |  |  | ...
Row 19: 29 | Iowa |  |  |  | ...
Row 20: 34 | Kansas |  |  |  | ...
Row 21: 15 | Kentucky |  | 4 | 4 | ...
Row 22: 18 | Louisiana |  |  |  | ...
Row 23: 23 | Maine |  |  |  | ...
Row 24: 7 | Maryland | 8 | 10 | 10 | ...
Row 25: 6 | Massachusetts | 10 | 16 | 16 | ...
Row 26: 

The print statements show the first four rows contain data that doesn't match what we need; these headers are actually the two levels of headers in the table. The print statement of the first few cells shows there are four 'header' rows: "Electionyear", "'88", " ", and "#".

Here is a screenshot of the top of the table from the webpage that shows the complexity of the table headers:

![Alt text](electoral_college_table_image.png)

Separate and organize the headers for the correct column names:

In [6]:
# first row of data and save as a header variable
header_row = all_data[0]
print(f"Header row: {header_row}\n")

Header row: ['Electionyear', '1788–1800', '1804–1900', '1904–2000', '2004–']



In [7]:
# save the 'Electionyear' value as a key variable
first_key = all_data[0][0]
first_key

'Electionyear'

In [8]:
# save the first header row: range of years in centuries
first_header = all_data[0][1::]
first_header

['1788–1800', '1804–1900', '1904–2000', '2004–']

In [9]:
# save the next header: election year cycles
second_header = all_data[1]
second_header

["'88",
 "'92",
 "'96'00",
 "'04'08",
 "'12",
 "'16",
 "'20",
 "'24'28",
 "'32",
 "'36'40",
 "'44",
 "'48",
 "'52'56",
 "'60",
 "'64",
 "'68",
 "'72",
 "'76'80",
 "'84'88",
 "'92",
 "'96'00",
 "'04",
 "'08",
 "'12'16'20'24'28",
 "'32'36'40",
 "'44'48",
 "'52'56",
 "'60",
 "'64'68",
 "'72'76'80",
 "'84'88",
 "'92'96'00",
 "'04'08",
 "'12'16'20",
 "'24'28"]

The fourth header, 'Total' electoral college votes, isn't necessary to keep within the table created for this project. The value can be derived from a sum of the specific year-range column. The following cells are making the header row more human-readable.

In [10]:
# might be useful to hardcode some of the century ranges for later use
range_1 = 1700
range_2 = 1800
range_3 = 1900
range_4 = 2000

# save a value to split the year values
split_val = 2

In [11]:
# function to strip apostrophes from year ranges
def strip_apost(text):
    new_text = text.replace("'", "")
    return new_text

# function to split the text by length
def split_by_length(text, length):
    return [text[i:i+length] for i in range(0, len(text), length)]

# create test data in list form
list_example = ["'76,'80,'84", "'44,'48'"]

# test the function to strip the apostrophes
new_string = strip_apost(list_example[0])
print(new_string)

76,80,84


In [12]:
# call function in loop to remove apostrophes, creates a single string for each header column
year_range = []
for element in second_header:
    new_val = strip_apost(element)
    year_range.append(new_val)
    # print(new_val)

print(year_range)

['88', '92', '9600', '0408', '12', '16', '20', '2428', '32', '3640', '44', '48', '5256', '60', '64', '68', '72', '7680', '8488', '92', '9600', '04', '08', '1216202428', '323640', '4448', '5256', '60', '6468', '727680', '8488', '929600', '0408', '121620', '2428']


In [13]:
# create list variable for the year values
split_years = []

# split the two digit year ranges into their own lists
for year in year_range:
    split_yr = split_by_length(year, split_val)
    split_years.append(split_yr)

print(split_years)

[['88'], ['92'], ['96', '00'], ['04', '08'], ['12'], ['16'], ['20'], ['24', '28'], ['32'], ['36', '40'], ['44'], ['48'], ['52', '56'], ['60'], ['64'], ['68'], ['72'], ['76', '80'], ['84', '88'], ['92'], ['96', '00'], ['04'], ['08'], ['12', '16', '20', '24', '28'], ['32', '36', '40'], ['44', '48'], ['52', '56'], ['60'], ['64', '68'], ['72', '76', '80'], ['84', '88'], ['92', '96', '00'], ['04', '08'], ['12', '16', '20'], ['24', '28']]


The values are currently strings, which makes it more difficult to complete the following task (which is to add the two-digit century to the beginning of each value.

In [14]:
# convert string values to integers
integer_year = []

for first_list in split_years:
    new_list = []
    for item in first_list:
        try:
            new_list.append(int(item))
        except ValueError:
            new_list.append(item)

    integer_year.append(new_list)

print(integer_year)

[[88], [92], [96, 0], [4, 8], [12], [16], [20], [24, 28], [32], [36, 40], [44], [48], [52, 56], [60], [64], [68], [72], [76, 80], [84, 88], [92], [96, 0], [4], [8], [12, 16, 20, 24, 28], [32, 36, 40], [44, 48], [52, 56], [60], [64, 68], [72, 76, 80], [84, 88], [92, 96, 0], [4, 8], [12, 16, 20], [24, 28]]


In [15]:
# add two digits to beginning of new two digit integers for the century

century_years = []

# range for 1700s (first three values)
for lists in integer_year[:3]:
    add_century = []
    for item in lists:
        new_item = item + 1700
        if new_item % 100 == 0:
            new_item = 1800
        add_century.append(new_item)
    century_years.append(add_century)

# range for 1800s (fourth to twenty-first values)
for lists in integer_year[3:21]:
    add_century = []
    for item in lists:
        new_item = item + 1800
        if new_item % 100 == 0:
            new_item = 1900
        add_century.append(new_item)
    century_years.append(add_century)

# range for 1900s (twenty-second to thirty-second values)
for lists in integer_year[21:32]:
    add_century = []
    for item in lists:
        new_item = item + 1900
        if new_item % 100 == 0:
            new_item = 2000
        add_century.append(new_item)
    century_years.append(add_century)

# range for 2000s (thirty-third to final value)
for lists in integer_year[32:]:
    add_century = []
    for item in lists:
        new_item = item + 2000
        add_century.append(new_item)
    century_years.append(add_century)

print(century_years)

[[1788], [1792], [1796, 1800], [1804, 1808], [1812], [1816], [1820], [1824, 1828], [1832], [1836, 1840], [1844], [1848], [1852, 1856], [1860], [1864], [1868], [1872], [1876, 1880], [1884, 1888], [1892], [1896, 1900], [1904], [1908], [1912, 1916, 1920, 1924, 1928], [1932, 1936, 1940], [1944, 1948], [1952, 1956], [1960], [1964, 1968], [1972, 1976, 1980], [1984, 1988], [1992, 1996, 2000], [2004, 2008], [2012, 2016, 2020], [2024, 2028]]


Now that the 'year' header looks better, we can now split the second header, 'states'.

In [16]:
# Start checking from row 3
data_start = 3

# create list variable for state names
state_col = []

# loop through the rows and get the second value: state name
# the first value signifies when the state was added to the union (not needed)
for i in range(2, len(all_data)):
    first_col_data = all_data[i][1]
    # print(first_col_data)
    state_col.append(first_col_data)

# delete the header rows and the final row (total votes)
del state_col[0:2]
del state_col[-1]

# verify the list contains only the state names
print(state_col)

['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut', 'D.C.', 'Delaware', 'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming']


In [24]:
# retrieve the votes for each year range for each state (in its own list)
electoral_votes = [row[1:] for row in all_data[data_start:] if row[0]]
# print(electoral_votes)

In [25]:
# create dictionary in order to create a csv using a dictionary format
state_data = {state_col[i]: electoral_votes[i] for i in range(len(state_col))}
# state_data

Create a csv with the electoral data and correct year ranges:

In [19]:
with open('data/electoral_data.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['State'] + century_years)  # Header
    for state in state_col:
        writer.writerow([state] + state_data[state])  # Data

print("\n✓ Saved to electoral_data.csv")


✓ Saved to electoral_data.csv


View the csv as a dataframe using pandas:

In [20]:
import pandas as pd
electoral_data = pd.read_csv("data/electoral_data.csv")

# convert NaN values to 0 for easier analysis
electoral_df = electoral_data.fillna(0)

# convert all float values to integer
electoral_df = electoral_df.apply(lambda column: column.astype("int64") if pd.api.types.is_float_dtype(column) else column)

electoral_df

,State,[1788],[1792],"[1796, 1800]","[1804, 1808]",[1812],[1816],[1820],"[1824, 1828]",[1832],...,"[1944, 1948]","[1952, 1956]",[1960],"[1964, 1968]","[1972, 1976, 1980]","[1984, 1988]","[1992, 1996, 2000]","[2004, 2008]","[2012, 2016, 2020]","[2024, 2028]"
Alabama,Alabama,0,0,0,0,0,0,3,5,7,...,11,11,11,10,9,9,9,9,9,9
Alaska,Alaska,0,0,0,0,0,0,0,0,0,...,0,0,3,3,3,3,3,3,3,3
Arizona,Arizona,0,0,0,0,0,0,0,0,0,...,4,4,4,5,6,7,8,10,11,11
Arkansas,Arkansas,0,0,0,0,0,0,0,0,0,...,9,8,8,6,6,6,6,6,6,6
California,California,0,0,0,0,0,0,0,0,0,...,25,32,32,40,45,47,54,55,55,54
Colorado,Colorado,0,0,0,0,0,0,0,0,0,...,6,6,6,6,7,8,8,9,9,10
Connecticut,Connecticut,7,9,9,9,9,9,9,8,8,...,8,8,8,8,8,8,8,7,7,7
D.C.,D.C.,0,0,0,0,0,0,0,0,0,...,0,0,0,3,3,3,3,3,3,3
Delaware,Delaware,3,3,3,3,4,4,4,3,3,...,3,3,3,3,3,3,3,3,3,3
Florida,Florida,0,0,0,0,0,0,0,0,0,...,8,10,10,14,17,21,25,27,29,30


Note: Most states' votes are 'winner takes all' (all electoral votes are cast to the popular vote). Maine and Nebraska use the congressional districting method (votes are split by the district popular vote).

In [21]:
# reset index
electoral_df = electoral_df.reset_index(drop=True)
electoral_df

,State,[1788],[1792],"[1796, 1800]","[1804, 1808]",[1812],[1816],[1820],"[1824, 1828]",[1832],...,"[1944, 1948]","[1952, 1956]",[1960],"[1964, 1968]","[1972, 1976, 1980]","[1984, 1988]","[1992, 1996, 2000]","[2004, 2008]","[2012, 2016, 2020]","[2024, 2028]"
0,Alabama,0,0,0,0,0,0,3,5,7,...,11,11,11,10,9,9,9,9,9,9
1,Alaska,0,0,0,0,0,0,0,0,0,...,0,0,3,3,3,3,3,3,3,3
2,Arizona,0,0,0,0,0,0,0,0,0,...,4,4,4,5,6,7,8,10,11,11
3,Arkansas,0,0,0,0,0,0,0,0,0,...,9,8,8,6,6,6,6,6,6,6
4,California,0,0,0,0,0,0,0,0,0,...,25,32,32,40,45,47,54,55,55,54
5,Colorado,0,0,0,0,0,0,0,0,0,...,6,6,6,6,7,8,8,9,9,10
6,Connecticut,7,9,9,9,9,9,9,8,8,...,8,8,8,8,8,8,8,7,7,7
7,D.C.,0,0,0,0,0,0,0,0,0,...,0,0,0,3,3,3,3,3,3,3
8,Delaware,3,3,3,3,4,4,4,3,3,...,3,3,3,3,3,3,3,3,3,3
9,Florida,0,0,0,0,0,0,0,0,0,...,8,10,10,14,17,21,25,27,29,30


In [22]:
# save as csv with updated index
electoral_df.to_csv("data/electoral_data.csv", index=False)

In [23]:
test = pd.read_csv("data/electoral_data.csv")
test

,State,[1788],[1792],"[1796, 1800]","[1804, 1808]",[1812],[1816],[1820],"[1824, 1828]",[1832],...,"[1944, 1948]","[1952, 1956]",[1960],"[1964, 1968]","[1972, 1976, 1980]","[1984, 1988]","[1992, 1996, 2000]","[2004, 2008]","[2012, 2016, 2020]","[2024, 2028]"
0,Alabama,0,0,0,0,0,0,3,5,7,...,11,11,11,10,9,9,9,9,9,9
1,Alaska,0,0,0,0,0,0,0,0,0,...,0,0,3,3,3,3,3,3,3,3
2,Arizona,0,0,0,0,0,0,0,0,0,...,4,4,4,5,6,7,8,10,11,11
3,Arkansas,0,0,0,0,0,0,0,0,0,...,9,8,8,6,6,6,6,6,6,6
4,California,0,0,0,0,0,0,0,0,0,...,25,32,32,40,45,47,54,55,55,54
5,Colorado,0,0,0,0,0,0,0,0,0,...,6,6,6,6,7,8,8,9,9,10
6,Connecticut,7,9,9,9,9,9,9,8,8,...,8,8,8,8,8,8,8,7,7,7
7,D.C.,0,0,0,0,0,0,0,0,0,...,0,0,0,3,3,3,3,3,3,3
8,Delaware,3,3,3,3,4,4,4,3,3,...,3,3,3,3,3,3,3,3,3,3
9,Florida,0,0,0,0,0,0,0,0,0,...,8,10,10,14,17,21,25,27,29,30
